# 04 — Gemma Basics

Load Gemma and run rhythm explanations.

**Three ways to run:**
1. **Cloud API** (fast) — set `USE_CLOUD = True` and point to a vLLM / Ollama / cloud endpoint
2. **Local GPU** (default) — 4-bit quantized on your GPU
3. **Local CPU** (automatic fallback) — slow (~1 min/response) but always works

First local load downloads ~3 GB. Subsequent loads use `~/.cache/huggingface/`.

In [ ]:
# ── Configuration ─────────────────────────────────────────────
# Flip USE_CLOUD to True and set your endpoint to skip local model loading.
# Or set env vars RYTMI_API_BASE_URL (and optionally RYTMI_API_KEY) instead.

USE_CLOUD = True

# Only needed when USE_CLOUD = True:
CLOUD_BASE_URL = "http://localhost:11434/v1"  # Ollama default
CLOUD_MODEL_ID = "gemma4:e4b"                 # ~5 GB VRAM as Q4; use "gemma4:e2b" for smaller

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown

from rytmi import load_audio, analyze
from rytmi.llm import load_model, load_cloud_model, generate, explain_rhythm, explain_all
from rytmi.prompts import ALL_QUESTIONS, format_analysis_prompt

In [ ]:
if USE_CLOUD:
    processor, model = load_cloud_model(model_id=CLOUD_MODEL_ID, base_url=CLOUD_BASE_URL)
    print(f"Cloud backend: {CLOUD_BASE_URL}  model={CLOUD_MODEL_ID}")
else:
    processor, model = load_model()  # profile="laptop", quantize=True by default

In [ ]:
if USE_CLOUD:
    print("Using cloud backend — no local VRAM used")
else:
    import torch
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved() / 1e9
        print(f"VRAM allocated: {allocated:.2f} GB")
        print(f"VRAM reserved:  {reserved:.2f} GB")
    else:
        print("Running on CPU")

## 2. Sanity check — plain generation

In [ ]:
response = generate(processor, model, "What is 4/4 time signature? Answer in one sentence.")
print(response)

## 3. Run the DSP pipeline

In [ ]:
audio = load_audio(Path("../data/samples/click_120bpm.wav"))
result = analyze(audio)
print(f"Tempo: {result.tempo:.1f} BPM | Beats: {len(result.beats.times)} | Onsets: {len(result.onsets.times)}")

## 4. Ask Gemma individual questions

In [ ]:
from rytmi.prompts import QUESTION_COUNTING, QUESTION_TIME_SIGNATURE, QUESTION_STYLE

for label, question in [("Counting", QUESTION_COUNTING),
                         ("Time signature", QUESTION_TIME_SIGNATURE),
                         ("Style", QUESTION_STYLE)]:
    print(f"\n--- {label} ---")
    resp = explain_rhythm(result, question=question, processor=processor, model=model)
    print(resp)

## 5. Run all questions

In [ ]:
all_responses = explain_all(result, processor=processor, model=model)

for key, response in all_responses.items():
    display(Markdown(f"### {key.replace('_', ' ').title()}\n\n{response}"))

## 6. Inspect the prompt

Useful for prompt engineering — see exactly what goes into the model.

In [ ]:
prompt = format_analysis_prompt(
    duration=result.audio.duration,
    tempo=result.tempo,
    n_beats=len(result.beats.times),
    n_onsets=len(result.onsets.times),
    beat_times=result.beats.times,
    ioi_ms=result.inter_onset_intervals,
    question=QUESTION_COUNTING,
)
print(prompt)

## 7. Try with your own audio

## Backend options

**Cloud** — set `USE_CLOUD = True` in the config cell and point to your endpoint:
```python
USE_CLOUD = True
CLOUD_BASE_URL = "http://localhost:8000/v1"  # or any OpenAI-compatible API
```

**Kaggle** — use the larger model on GPU:
```python
USE_CLOUD = False
processor, model = load_model(profile="kaggle", quantize=False)
```